In [1]:
# 1. Systeem dependencies
!apt-get update && apt-get install -y zstd

# 2. Ollama installeren
!curl -fsSL https://ollama.com/install.sh | sh

# 3. CrewAI installeren (we negeren de errors van de Google-pakketten)
!pip install -q --no-warn-conflicts crewai langchain_community

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,855 kB] 
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:13 https://ppa.launc

In [2]:
import crewai
import langchain_community
print("CrewAI is succesvol geladen!")

CrewAI is succesvol geladen!


In [3]:
import os
import subprocess
import time

# 1. Installeer zstd en Ollama (met forcering van het pad)
print("Bezig met installeren van dependencies...")
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Definieer het volledige pad naar ollama
# De installer zet hem meestal in /usr/local/bin/ollama
OLLAMA_PATH = "/usr/local/bin/ollama"

# 3. Start de server op de achtergrond
print("Ollama server opstarten...")
subprocess.Popen([OLLAMA_PATH, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15) # Geef de server tijd

# 4. Pull het model met het volledige pad
print("Model downloaden (kan even duren)...")
subprocess.run([OLLAMA_PATH, "pull", "llama3"])

# 5. Controleer of het model er staat
print("\nGeïnstalleerde modellen:")
subprocess.run([OLLAMA_PATH, "list"])

Bezig met installeren van dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease          
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading 

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  30 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  59 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   3% ▕                  ▏ 133 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   5% ▕                  ▏ 217 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   


Geïnstalleerde modellen:
NAME             ID              SIZE      MODIFIED               
llama3:latest    365c0bd3c000    4.7 GB    Less than a second ago    


pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

In [4]:
import os
from crewai import Agent, Task, Crew

# 1. Fake OpenAI key (verplicht voor CrewAI)
os.environ["OPENAI_API_KEY"] = "sk-ollama"

# 2. Configureer de Agent met de 'openai' provider stijl, maar wijs naar Ollama
# We gebruiken 'ollama/' als prefix voor de modelnaam
test_agent = Agent(
    role='Kaggle Expert',
    goal='Laat zien dat de 404 error weg is.',
    backstory='Ik ben een AI die lokaal draait.',
    # De magie zit hier:
    llm="ollama/llama3", 
    # We vertellen CrewAI expliciet waar de lokale server staat
    base_url="http://localhost:11434/v1", 
    verbose=True,
    allow_delegation=False
)

# 3. Simpele taak
test_task = Task(
    description='Zeg alleen: "De 404 is opgelost!"',
    expected_output='Een korte bevestiging.',
    agent=test_agent
)

crew = Crew(agents=[test_agent], tasks=[test_task])

print("\n--- START TEST ---")
result = crew.kickoff()
print(result)


--- START TEST ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Kaggle Expert                                                                                           │
│                                                                                                                 │
│  Task: Zeg alleen: "De 404 is opgelost!"                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Kaggle Expert                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  De 404 is opgelost!                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

De 404 is opgelost!


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                       

In [ ]:
import os
from crewai import Agent, Task, Crew

# 1. Configuratie (Fake key en Lokaal Model)
os.environ["OPENAI_API_KEY"] = "sk-agile-team"

# We definiëren de LLM configuratie één keer voor hergebruik
# Zorg dat je exact deze modelnaam hebt 'llama3'
local_llm_config = {
    "model": "ollama/llama3",
    "base_url": "http://localhost:11434/v1"
}

# ==========================================
# 2. Definieer de Agents (De Crew Members)
# ==========================================

# Agent 1: De Ontwerper (UI/UX Designer)
designer = Agent(
    role='UI/UX Designer',
    goal='Ontwerp een minimalistische en moderne "Hello World" React website.',
    backstory='Je bent een expert in gebruikersinterface-ontwerp. Je houdt van strakke layouts, Tailwind CSS en minimalistische esthetiek.',
    llm=local_llm_config["model"],
    base_url=local_llm_config["base_url"],
    allow_delegation=False,
    verbose=True
)

# Agent 2: De Ontwikkelaar (React Developer)
developer = Agent(
    role='React Developer',
    goal='Bouw de website in React, gebaseerd op het ontwerp.',
    backstory='Je bent een senior frontend engineer. Je schrijft schone, efficiënte React (JSX/TSX) code en gebruikt functionele componenten.',
    llm=local_llm_config["model"],
    base_url=local_llm_config["base_url"],
    allow_delegation=False,
    verbose=True
)

# Agent 3: De Tester (QA Engineer)
tester = Agent(
    role='QA Engineer',
    goal='Test de opgeleverde React code met Playwright.',
    backstory='Je bent een expert in geautomatiseerd testen. Je schrijft robuuste Playwright testscripts in TypeScript die controleren op functionaliteit en UI elementen.',
    llm=local_llm_config["model"],
    base_url=local_llm_config["base_url"],
    allow_delegation=False,
    verbose=True
)

# ==========================================
# 3. Definieer de Taken (De Sprint Backlog)
# ==========================================

# Taak 1: Ontwerp specificatie
task_design = Task(
    description=(
        "Bedenk een ontwerp voor een simpele 'Hello World' React website. "
        "Beschrijf de layout, het kleurenschema (bijv. light/dark mode), "
        "en de exacte tekst die op het scherm moet staan. Stel voor welke "
        "CSS library (bijv. Tailwind) gebruikt moet worden."
    ),
    expected_output="Een ontwerpspecificatie document met layout details en styling voorstellen.",
    agent=designer
)

# Taak 2: React Code genereren
task_development = Task(
    description=(
        "Bouw de React component(en) gebaseerd op de ontwerpspecificatie van de Designer. "
        "Schrijf de volledige code voor een `App.js` (of `App.tsx`) bestand. "
        "Zorg ervoor dat de code schoon is en de 'Hello World' tekst correct weergeeft."
    ),
    expected_output="De volledige JSX/TSX code voor de React applicatie.",
    agent=developer,
    context=[task_design] # De Developer heeft de output van de Designer nodig
)

# Taak 3: Playwright Testscript schrijven
task_testing = Task(
    description=(
        "Schrijf een Playwright testscript om de React applicatie te valideren. "
        "Het script moet controleren of de pagina laadt, of de 'Hello World' tekst "
        "aanwezig is, en of de basis styling (zoals de achtergrondkleur) klopt. "
        "Gebruik de opgeleverde React code om te bepalen op welke selectors je moet testen."
    ),
    expected_output="Een volledig Playwright testscript in TypeScript.",
    agent=tester,
    context=[task_development] # De Tester heeft de code van de Developer nodig
)

# ==========================================
# 4. Start de Agile Crew (De Sprint)
# ==========================================

agile_crew = Crew(
    agents=[designer, developer, tester],
    tasks=[task_design, task_development, task_testing],
    verbose=True, # Laat de gedachten van de agents zien
    process="sequential" # Cruciaal: Ze werken na elkaar
)

print("\n--- AGILE SPRINT START: 'HELLO WORLD' IN REACT ---")
try:
    result = agile_crew.kickoff()
    print("\n" + "="*80)
    print("FINAL SPRINT DELIVERABLE (TEST SCRIPT):")
    print("\n" + result)
    print("="*80)
except Exception as e:
    print(f"\n❌ Er ging iets mis tijdens de sprint: {e}")


--- AGILE SPRINT START: 'HELLO WORLD' IN REACT ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a4effed9-11a8-4670-be15-ec94e9139cb2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Bedenk een ontwerp voor een simpele 'Hello World' React website. Beschrijf de layout, het kleurenschema  │
│  (bijv. light/dark mode), en de exacte tekst die op het scherm moet staan. Stel voor welke CSS library (bijv.   │
│  Tailwind) gebruikt moet worden.                                                                                │
│  ID: 16f5749d-b46b-4e51-ba2f-8c15202875b4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: UI/UX Designer                                                                                          │
│                                                                                                                 │
│  Task: Bedenk een ontwerp voor een simpele 'Hello World' React website. Beschrijf de layout, het kleurenschema  │
│  (bijv. light/dark mode), en de exacte tekst die op het scherm moet staan. Stel voor welke CSS library (bijv.   │
│  Tailwind) gebruikt moet worden.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: UI/UX Designer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Minimalistic "Hello World" React Website Design Specification**                                              │
│                                                                                                                 │
│  **Layout:**                                                                                                    │
│                                                                                                                 │
│  The website will feature a single page layout with a minimalistic design. The main content area will be        │
│  centered on the page, with a fixed width of 800px. The background will be a clean and neutral white            │
│  (#FFFFFF).                                                                                                     │
│                                                                                                                 │
│  ```                                                                                                            │
│  <header>                                                                                                       │
│    <nav>                                                                                                        │
│      <ul>                                                                                                       │
│        <li><a href="#home">Home</a></li>                                                                        │
│      </ul>                                                                                                      │
│    </nav>                                                                                                       │
│  </header>                                                                                                      │
│  <main>                                                                                                         │
│    <section id="home">                                                                                          │
│      <h1>Hello World!</h1>                                                                                      │
│      <p>This is a minimalistic "Hello World" React website.</p>                                                 │
│    </section>                                                                                                   │
│  </main>                                                                                                        │
│  <footer>                                                                                                       │
│    <p>&copy; 2023 Minimalistic "Hello World" React Website</p>                                                  │
│  </footer>                                                                                                      │
│  ```                                                                                                            │
│                                                                                                                 │
│  **Styling:**                                                                                                   │
│                                                                                                                 │
│  To achieve a modern and minimalist look, we will use T

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Bedenk een ontwerp voor een simpele 'Hello World' React website. Beschrijf de layout, het kleurenschema  │
│  (bijv. light/dark mode), en de exacte tekst die op het scherm moet staan. Stel voor welke CSS library (bijv.   │
│  Tailwind) gebruikt moet worden.                                                                                │
│  Agent: UI/UX Designer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Bouw de React component(en) gebaseerd op de ontwerpspecificatie van de Designer. Schrijf de volledige    │
│  code voor een `App.js` (of `App.tsx`) bestand. Zorg ervoor dat de code schoon is en de 'Hello World' tekst     │
│  correct weergeeft.                                                                                             │
│  ID: 565bc5a5-e057-401e-9b2e-bda9850b7404                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: React Developer                                                                                         │
│                                                                                                                 │
│  Task: Bouw de React component(en) gebaseerd op de ontwerpspecificatie van de Designer. Schrijf de volledige    │
│  code voor een `App.js` (of `App.tsx`) bestand. Zorg ervoor dat de code schoon is en de 'Hello World' tekst     │
│  correct weergeeft.                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯